# Wang-Resnick Visual Notebook

Notebook для интерактивной генерации графиков по коду дипломного проекта. Ячейки ниже повторяют основные сценарии CLI-скриптов, но оставляют управление параметрами и визуализацией внутри Jupyter.

Изображения сохраняются в `results/notebook/figures/`.

## 1. Инициализация

In [2]:
%matplotlib inline

import contextlib
import io
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from IPython.display import Image, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUT_DIR = ROOT / "results" / "notebook"
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TRUNCATION_K = 10_000

ROOT, FIG_DIR

(PosixPath('/home/waimer/Projects/mipt/master-deg-project'),
 PosixPath('/home/waimer/Projects/mipt/master-deg-project/results/notebook/figures'))

In [3]:
from main import make_default_graph
from scripts.run_nk_experiment import (
    ExperimentConfig as NkConfig,
    empirical_summary as nk_empirical_summary,
    simulate_nk_trajectories,
    theoretical_moments as nk_theoretical_moments,
)
from scripts.run_pn_experiment import ExperimentConfig as PnConfig, build_snapshots as build_pn_snapshots
from scripts.run_sk_experiment import ExperimentConfig as SkConfig, simulate_trajectories as simulate_sk_trajectories
from scripts.run_sk_kstar_experiment import (
    ExperimentConfig as KstarConfig,
    simulate_final_samples as simulate_kstar_samples,
)

## 2. Эволюция Графа

Эта ячейка строит сетку снимков графа для разных `p` и выбранных шагов эволюции. Черные вершины имеют сообщение, серые - нет.

In [4]:
def graph_snapshots(
    *,
    p: float,
    c: float,
    checkpoints: tuple[int, ...],
    seed: int = SEED,
    truncation_k: int = TRUNCATION_K,
) -> dict[int, nx.MultiDiGraph]:
    graph = make_default_graph(c=c, p=p, delta_in=1, delta_out=1, rng=np.random.default_rng(seed))
    snapshots = {0: graph._graph.copy()}
    for step in range(1, max(checkpoints) + 1):
        with contextlib.redirect_stdout(io.StringIO()):
            graph.evolute(rounds=1, k=truncation_k)
        if step in checkpoints:
            snapshots[step] = graph._graph.copy()
    return snapshots


def draw_snapshot(ax: plt.Axes, graph: nx.MultiDiGraph, *, title: str, seed: int) -> None:
    nodes = list(graph.nodes())
    if not nodes:
        ax.set_axis_off()
        return

    pos = nx.spring_layout(graph, seed=seed, iterations=250)
    colors = ["#222222" if graph.nodes[node].get("message", False) else "#b8b8b8" for node in nodes]

    nx.draw_networkx_nodes(
        graph,
        pos,
        ax=ax,
        node_color=colors,
        edgecolors="white",
        linewidths=0.6,
        node_size=90,
    )
    nx.draw_networkx_edges(
        graph,
        pos,
        ax=ax,
        arrows=True,
        arrowsize=8,
        width=0.8,
        alpha=0.45,
        edge_color="#666666",
        connectionstyle="arc3,rad=0.05",
    )
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()


def plot_evolution_grid(
    *,
    p_values: tuple[float, ...] = (0.1, 0.5, 0.9, 0.99),
    c: float = 0.1,
    checkpoints: tuple[int, ...] = (0, 1, 5, 10),
    seed: int = SEED,
    output_name: str = "evolution_grid.png",
) -> plt.Figure:
    fig, axes = plt.subplots(
        len(p_values),
        len(checkpoints),
        figsize=(3.2 * len(checkpoints), 2.8 * len(p_values)),
        layout="constrained",
    )
    axes = np.atleast_2d(axes)

    for row, p_value in enumerate(p_values):
        snapshots = graph_snapshots(p=p_value, c=c, checkpoints=checkpoints, seed=seed + row)
        for col, step in enumerate(checkpoints):
            draw_snapshot(
                axes[row, col],
                snapshots[step],
                title=f"p={p_value:g}, k={step}",
                seed=seed + 100 * row + col,
            )

    fig.suptitle(f"Эволюция графа при c={c:g}")
    fig.savefig(FIG_DIR / output_name, dpi=180)
    return fig

In [5]:
fig = plot_evolution_grid(c=0.1, checkpoints=(0, 1, 5, 10))

## 3. Проверка `N_k`

Сравнение эмпирического среднего/дисперсии с теоретическими значениями. Для быстрого интерактивного запуска используются умеренные `runs` и `steps`; для дипломных рисунков увеличьте их до нужных значений.

In [6]:
def plot_nk_theory_check(
    *,
    p: float = 0.5,
    c: float = 0.2,
    runs: int = 100,
    steps: int = 200,
    seed: int = SEED,
    output_name: str = "nk_theory_check.png",
) -> tuple[plt.Figure, np.ndarray]:
    config = NkConfig(p=p, c=c, runs=runs, steps=steps, seed=seed, truncation_k=TRUNCATION_K)
    trajectories = simulate_nk_trajectories(config)
    _lambda_prefix, theoretical_mean, theoretical_var = nk_theoretical_moments(config)
    empirical_mean, _empirical_std, empirical_var = nk_empirical_summary(trajectories)
    x = np.arange(steps + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")
    axes[0].plot(x, theoretical_mean, color="black", linewidth=1.8, label="Theory")
    axes[0].plot(x, empirical_mean, color="tab:blue", linewidth=1.4, label="Simulation")
    axes[0].set_title(r"$E[N_k]$")
    axes[0].set_xlabel("k")
    axes[0].set_ylabel(r"$N_k$")
    axes[0].grid(alpha=0.25)
    axes[0].legend()

    axes[1].plot(x, theoretical_var, color="black", linewidth=1.8, label="Theory")
    axes[1].plot(x, empirical_var, color="tab:orange", linewidth=1.4, label="Simulation")
    axes[1].set_title(r"$Var(N_k)$")
    axes[1].set_xlabel("k")
    axes[1].set_ylabel(r"$Var(N_k)$")
    axes[1].grid(alpha=0.25)
    axes[1].legend()

    fig.suptitle(f"N_k check: p={p:g}, c={c:g}, runs={runs}")
    fig.savefig(FIG_DIR / output_name, dpi=180)
    return fig, trajectories

In [7]:
fig, nk_trajectories = plot_nk_theory_check(p=0.5, c=0.2, runs=100, steps=200)

## 4. `|S_k|` И Относительный Охват

Средние траектории числа информированных узлов и доли `|S_k| / N_k`.

In [8]:
def plot_sk_ensemble(
    *,
    p: float = 0.5,
    c: float = 0.2,
    runs: int = 100,
    steps: int = 200,
    seed: int = SEED,
    output_name: str = "sk_ensemble.png",
) -> tuple[plt.Figure, tuple[np.ndarray, np.ndarray, np.ndarray]]:
    config = SkConfig(p=p, c=c, runs=runs, steps=steps, seed=seed, truncation_k=TRUNCATION_K)
    n_trajectories, s_trajectories, ratios = simulate_sk_trajectories(config)
    x = np.arange(steps + 1)

    s_mean = s_trajectories.mean(axis=0)
    s_std = s_trajectories.std(axis=0, ddof=1)
    ratio_mean = ratios.mean(axis=0)
    ratio_std = ratios.std(axis=0, ddof=1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")
    axes[0].plot(x, s_mean, color="tab:green", linewidth=1.8, label="mean")
    axes[0].fill_between(x, s_mean - s_std, s_mean + s_std, color="tab:green", alpha=0.18)
    axes[0].set_title(r"$|S_k|$")
    axes[0].set_xlabel("Шаг эволюции")
    axes[0].set_ylabel(r"$|S_k|$")
    axes[0].grid(alpha=0.25)

    axes[1].plot(x, ratio_mean, color="tab:blue", linewidth=1.8, label="mean")
    axes[1].fill_between(
        x,
        np.clip(ratio_mean - ratio_std, 0.0, 1.0),
        np.clip(ratio_mean + ratio_std, 0.0, 1.0),
        color="tab:blue",
        alpha=0.18,
    )
    axes[1].set_title(r"$|S_k| / N_k$")
    axes[1].set_xlabel("Шаг эволюции")
    axes[1].set_ylabel(r"$|S_k| / N_k$")
    axes[1].set_ylim(0.0, 1.05)
    axes[1].grid(alpha=0.25)

    fig.suptitle(f"Information spreading: p={p:g}, c={c:g}, runs={runs}")
    fig.savefig(FIG_DIR / output_name, dpi=180)
    return fig, (n_trajectories, s_trajectories, ratios)

In [9]:
fig, sk_samples = plot_sk_ensemble(p=0.5, c=0.2, runs=100, steps=200)

## 5. Случайный Момент `K*`

Распределение итогового охвата при `K* ~ Poisson(nu T*)`.

In [10]:
def plot_kstar_distribution(
    *,
    p: float = 0.5,
    c: float = 0.2,
    runs: int = 200,
    nu: float = 1.0,
    t_star: float = 200.0,
    seed: int = SEED,
    output_name: str = "kstar_distribution.png",
) -> tuple[plt.Figure, tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    config = KstarConfig(p=p, c=c, runs=runs, seed=seed, truncation_k=TRUNCATION_K, nu=nu, t_star=t_star)
    k_star, n_kstar, s_kstar, ratio_kstar = simulate_kstar_samples(config)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.0), layout="constrained")
    axes[0].hist(k_star, bins=min(25, max(8, int(np.sqrt(runs)))), color="#b0b0b0", edgecolor="white")
    axes[0].set_title(r"$K^*$")
    axes[0].set_xlabel("steps")
    axes[0].set_ylabel("count")

    axes[1].hist(s_kstar, bins=min(25, max(8, int(np.sqrt(runs)))), color="#73c6b6", edgecolor="white")
    axes[1].set_title(r"$|S_{K^*}|$")
    axes[1].set_xlabel("informed nodes")

    axes[2].hist(ratio_kstar, bins=np.linspace(0.0, 1.0, 26), color="#7fb3d5", edgecolor="white")
    axes[2].set_title(r"$|S_{K^*}| / N_{K^*}$")
    axes[2].set_xlabel("coverage")
    axes[2].set_xlim(0.0, 1.0)

    for ax in axes:
        ax.grid(alpha=0.25, axis="y")

    fig.suptitle(f"K* experiment: p={p:g}, c={c:g}, runs={runs}")
    fig.savefig(FIG_DIR / output_name, dpi=180)
    return fig, (k_star, n_kstar, s_kstar, ratio_kstar)

In [11]:
fig, kstar_samples = plot_kstar_distribution(p=0.5, c=0.2, runs=200, nu=1.0, t_star=200.0)

## 6. Локальная Вероятность `p_n`

Scatter plot теоретических значений `p_n` и эмпирических оценок из фиксированных состояний графа.

In [12]:
def plot_pn_scatter(
    *,
    p_values: tuple[float, ...] = (0.1, 0.5, 0.9),
    c_values: tuple[float, ...] = (0.1, 0.2, 0.3),
    snapshot_steps: tuple[int, ...] = (10, 30, 50),
    base_runs: int = 3,
    replications: int = 200,
    seed: int = SEED,
    output_name: str = "pn_scatter.png",
) -> tuple[plt.Figure, list]:
    config = PnConfig(
        p_values=p_values,
        c_values=c_values,
        snapshot_steps=snapshot_steps,
        base_runs=base_runs,
        replications=replications,
        seed=seed,
        truncation_k=TRUNCATION_K,
    )
    rows = build_pn_snapshots(config)
    theory = np.array([row.p_theory for row in rows], dtype=np.float64)
    empirical = np.array([row.p_empirical for row in rows], dtype=np.float64)

    fig, ax = plt.subplots(figsize=(5.4, 5.4), layout="constrained")
    ax.scatter(theory, empirical, s=24, alpha=0.75, color="tab:blue")
    ax.plot([0.0, 1.0], [0.0, 1.0], color="black", linewidth=1.2, linestyle="--")
    ax.set_xlabel(r"Theoretical $p_n$")
    ax.set_ylabel(r"Empirical $\widehat{p}_n$")
    ax.set_title(r"Local probability check")
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.25)
    fig.savefig(FIG_DIR / output_name, dpi=180)
    return fig, rows

In [13]:
fig, pn_rows = plot_pn_scatter(replications=200)

## 7. SNAP Fitting (Опционально)

Перед запуском выполните `make snap-download`. Ячейка ниже выключена по умолчанию, так как fitting и симуляции занимают заметное время.

In [ ]:
RUN_REAL_FIT = False

if RUN_REAL_FIT:
    from scripts.run_collegemsg_fit import CollegeMsgFitConfig, run as run_fit

    fit_output_dir = run_fit(
        CollegeMsgFitConfig(
            data_path=str(ROOT / "data" / "CollegeMsg.txt.gz"),
            output_dir=str(OUTPUT_DIR / "collegemsg_fit"),
            dataset_label="CollegeMsg",
            runs=20,
            seed=SEED,
        )
    )

    for image_name in (
        "degree_tail_compare.png",
        "message_coverage_compare.png",
        "angular_density_compare.png",
        "empirical_dynamics.png",
    ):
        image_path = fit_output_dir / image_name
        if image_path.exists():
            display(Image(filename=str(image_path)))

## 8. Список Сгенерированных Изображений

In [15]:
sorted(FIG_DIR.glob("*.png"))

[PosixPath('/home/waimer/Projects/mipt/master-deg-project/results/notebook/figures/evolution_grid.png'),
 PosixPath('/home/waimer/Projects/mipt/master-deg-project/results/notebook/figures/kstar_distribution.png'),
 PosixPath('/home/waimer/Projects/mipt/master-deg-project/results/notebook/figures/nk_theory_check.png'),
 PosixPath('/home/waimer/Projects/mipt/master-deg-project/results/notebook/figures/pn_scatter.png'),
 PosixPath('/home/waimer/Projects/mipt/master-deg-project/results/notebook/figures/sk_ensemble.png')]